In [ ]:
import os
os.environ["TORCH_CUDA_ARCH_LIST"] = "9.0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"


In [ ]:
import os, sys, time, json, datetime
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
torch.backends.cudnn.benchmark = False

from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from PIL import Image

import segmentation_models_pytorch as smp
import albumentations as A


In [ ]:
from dataclasses import dataclass, field

@dataclass
class TrainConfig:
    # Detection des points d'intersection 5mm via PNG masks (alignement garanti).

    project_root: str = r"C:\Users\v\Desktop\ECGPerturb-main\data"
    image_dir: str = ""
    mask_dir: str = ""
    output_dir: str = ""

    mask_type: str = "mask_grid_intersections.png"

    encoder_name: str = "resnet34"
    encoder_weights: str = "imagenet"
    in_channels: int = 3
    num_classes: int = 1

    img_height: int = 1024
    img_width:  int = 1024

    batch_size: int = 4
    num_epochs: int = 30
    learning_rate: float = 1e-4
    weight_decay:  float = 1e-5
    num_workers: int = 0
    pin_memory:  bool = True

    loss_type: str = "bce_dice"
    bce_weight: float = 0.5
    scheduler_patience: int = 5
    scheduler_factor:  float = 0.5
    early_stop_patience: int = 15

    train_sources: list = field(default_factory=lambda: ["ECG_031", "ECG_032"])
    val_sources:   list = field(default_factory=lambda: ["ECG_033"])

    device: str = ""
    seed: int = 42
    save_every_n_epochs:       int = 10
    log_images_every_n_epochs: int = 5

    def __post_init__(self):
        if not self.image_dir:
            self.image_dir = os.path.join(self.project_root, "output_augmentation", "images")
        if not self.mask_dir:
            self.mask_dir = os.path.join(self.project_root, "output_augmentation", "masks")
        if not self.output_dir:
            self.output_dir = os.path.join(self.project_root, "training", "runs_intersection")
        if not self.device:
            self.device = "cuda" if torch.cuda.is_available() else "cpu"

        if self.device == "cuda":
            print(f"[OK] GPU detecte : {torch.cuda.get_device_name(0)}")
            vram = torch.cuda.get_device_properties(0).total_memory / 1e9
            print(f"   VRAM: {vram:.1f} GB")
        else:
            print("[!] Mode CPU actif - performances limitees a 1024x1024.")

cfg = TrainConfig()


In [ ]:
import cv2

class ECGGridDataset(Dataset):
    # Dataset (image augmentee, mask_grid_intersections.png) deja aligne par P2.
    # Le mask est dilate apres resize pour que les points (rendus tres petits en
    # native) restent visibles a 1024x1024 (sinon ils disparaissent au resize).

    def __init__(self, image_dir, mask_dir, mask_type="mask_grid_intersections.png",
                 source_prefixes=None, img_height=1024, img_width=1024,
                 dilate_kernel=5, augment=False):
        self.image_dir = image_dir
        self.mask_dir  = mask_dir
        self.mask_type = mask_type
        self.img_height, self.img_width = img_height, img_width
        self.dilate_kernel = dilate_kernel

        self.samples = []
        for fname in sorted(os.listdir(image_dir)):
            if not fname.endswith(".webp"):
                continue
            if source_prefixes and not any(fname.startswith(p) for p in source_prefixes):
                continue
            stem = os.path.splitext(fname)[0]
            mask_path = os.path.join(mask_dir, stem, mask_type)
            if os.path.exists(mask_path):
                self.samples.append({
                    "image_path": os.path.join(image_dir, fname),
                    "mask_path":  mask_path,
                    "stem":       stem,
                })

        if augment:
            self.transform = A.Compose([
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.3),
                A.ColorJitter(brightness=0.15, contrast=0.15,
                              saturation=0.1, hue=0.05, p=0.5),
                A.GaussNoise(p=0.2),
            ])
        else:
            self.transform = None

        print(f"  -> {len(self.samples)} paires trouvees (mask: {mask_type},"
              f" sources: {source_prefixes or 'toutes'},"
              f" dilate={dilate_kernel}x{dilate_kernel})")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img = Image.open(s["image_path"]).convert("RGB")
        img = img.resize((self.img_width, self.img_height), Image.BILINEAR)
        img_np = np.array(img, dtype=np.float32) / 255.0

        mask = Image.open(s["mask_path"]).convert("L")
        mask = mask.resize((self.img_width, self.img_height), Image.NEAREST)
        mask_np = np.array(mask, dtype=np.float32) / 255.0

        # Dilatation pour que les points survivent au resize (sinon disparaissent)
        if self.dilate_kernel and self.dilate_kernel > 1:
            kernel = np.ones((self.dilate_kernel, self.dilate_kernel), np.uint8)
            mask_np = cv2.dilate(mask_np, kernel, iterations=1)

        if self.transform:
            t = self.transform(image=img_np, mask=mask_np)
            img_np, mask_np = t["image"], t["mask"]

        img_tensor  = torch.from_numpy(img_np).permute(2, 0, 1).float()
        mask_tensor = torch.from_numpy(mask_np).unsqueeze(0).float()
        return img_tensor, mask_tensor


In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__(); self.smooth = smooth
    def forward(self, pred, target):
        ps = torch.sigmoid(pred)
        inter = (ps * target).sum(dim=(2, 3))
        union = ps.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
        return 1 - ((2 * inter + self.smooth) / (union + self.smooth)).mean()

class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5):
        super().__init__()
        self.bce, self.dice, self.bce_weight = nn.BCEWithLogitsLoss(), DiceLoss(), bce_weight
    def forward(self, pred, target):
        return self.bce_weight * self.bce(pred, target) + (1 - self.bce_weight) * self.dice(pred, target)

def get_loss(loss_type, bce_weight=0.5):
    return {"bce": nn.BCEWithLogitsLoss(),
            "dice": DiceLoss(),
            "bce_dice": BCEDiceLoss(bce_weight)}[loss_type]


def compute_metrics(pred, target, threshold=0.5):
    with torch.no_grad():
        pb = (torch.sigmoid(pred) > threshold).float()
        inter = (pb * target).sum(dim=(2, 3))
        ps = pb.sum(dim=(2, 3)); ts = target.sum(dim=(2, 3))
        dice = (2*inter + 1e-6) / (ps + ts + 1e-6)
        iou  = (inter + 1e-6) / (ps + ts - inter + 1e-6)
        rec  = (inter + 1e-6) / (ts + 1e-6)
        prec = (inter + 1e-6) / (ps + 1e-6)
    return {"dice": dice.mean().item(), "iou": iou.mean().item(),
            "precision": prec.mean().item(), "recall": rec.mean().item()}


def save_prediction_grid(images, masks_true, masks_pred, save_path, n=4):
    # Affiche image | GT | sigmoid continu (cmap=hot autoscale) | overlay binarise.
    n = min(n, images.shape[0])
    fig, axes = plt.subplots(n, 4, figsize=(16, 4*n))
    if n == 1: axes = axes[np.newaxis, :]
    for i in range(n):
        img = images[i].cpu().permute(1, 2, 0).numpy()
        mt  = masks_true[i, 0].cpu().numpy()
        mp_continuous = torch.sigmoid(masks_pred[i, 0]).cpu().numpy()
        mp_binary     = (mp_continuous > 0.5).astype(float)

        axes[i, 0].imshow(img);                                 axes[i, 0].set_title("Image augmentee", fontsize=9);                  axes[i, 0].axis("off")
        axes[i, 1].imshow(mt, cmap="gray", vmin=0, vmax=1);     axes[i, 1].set_title("Intersections GT", fontsize=9);                axes[i, 1].axis("off")
        axes[i, 2].imshow(mp_continuous, cmap="hot", vmin=0,
                          vmax=max(mp_continuous.max(), 1e-6)); axes[i, 2].set_title(f"Pred sigmoid (max={mp_continuous.max():.2f})", fontsize=9); axes[i, 2].axis("off")
        ov = img.copy()
        tp = (mp_binary > 0.5) & (mt > 0.5); fp = (mp_binary > 0.5) & (mt < 0.5); fn = (mp_binary < 0.5) & (mt > 0.5)
        ov[tp] = [0, 1, 0]; ov[fp] = [1, 0, 0]; ov[fn] = [0, 0, 1]
        axes[i, 3].imshow(ov);                                  axes[i, 3].set_title("Overlay (V=TP, R=FP, B=FN)", fontsize=9);     axes[i, 3].axis("off")
    plt.tight_layout(); plt.savefig(save_path, dpi=100, bbox_inches="tight"); plt.close()


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train(); total_loss, n = 0, 0
    tm = {"dice": 0, "iou": 0, "precision": 0, "recall": 0}
    pbar = tqdm(loader, desc="  Train", leave=False)
    for images, masks in pbar:
        images, masks = images.to(device), masks.to(device)
        pred = model(images); loss = criterion(pred, masks)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        m = compute_metrics(pred, masks)
        total_loss += loss.item()
        for k in tm: tm[k] += m[k]
        n += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}", dice=f"{m['dice']:.3f}")
    return total_loss / max(n, 1), {k: v / max(n, 1) for k, v in tm.items()}

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval(); total_loss, n = 0, 0
    tm = {"dice": 0, "iou": 0, "precision": 0, "recall": 0}
    for images, masks in tqdm(loader, desc="  Val  ", leave=False):
        images, masks = images.to(device), masks.to(device)
        pred = model(images); loss = criterion(pred, masks)
        m = compute_metrics(pred, masks)
        total_loss += loss.item()
        for k in tm: tm[k] += m[k]
        n += 1
    return total_loss / max(n, 1), {k: v / max(n, 1) for k, v in tm.items()}


def _plot_training_curves(history, save_path):
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
    e = range(1, len(history["train_loss"]) + 1)
    a1.plot(e, history["train_loss"], "b-", label="Train")
    a1.plot(e, history["val_loss"],   "r-", label="Val")
    a1.set_xlabel("Epoch"); a1.set_ylabel("Loss"); a1.set_title("Loss"); a1.legend(); a1.grid(True, alpha=0.3)
    a2.plot(e, history["train_dice"], "b-",  label="Train Dice")
    a2.plot(e, history["val_dice"],   "r-",  label="Val Dice")
    a2.plot(e, history["train_iou"],  "b--", alpha=0.5, label="Train IoU")
    a2.plot(e, history["val_iou"],    "r--", alpha=0.5, label="Val IoU")
    a2.set_xlabel("Epoch"); a2.set_ylabel("Score"); a2.set_title("Dice & IoU"); a2.legend(); a2.grid(True, alpha=0.3)
    plt.tight_layout(); plt.savefig(save_path, dpi=150, bbox_inches="tight"); plt.close()


def train(cfg: TrainConfig):
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    run_dir = os.path.join(cfg.output_dir, f"run_{timestamp}")
    os.makedirs(os.path.join(run_dir, "checkpoints"), exist_ok=True)
    os.makedirs(os.path.join(run_dir, "visualizations"), exist_ok=True)

    cfg_dict = {k: (v if isinstance(v, (int, float, bool, list, type(None))) else str(v))
                for k, v in cfg.__dict__.items()}
    with open(os.path.join(run_dir, "config.json"), "w") as f:
        json.dump(cfg_dict, f, indent=2)

    print(f"\n{'='*60}\n  Entrainement U-Net -- Detection intersections (PNG)\n{'='*60}")
    print(f"  Device      : {cfg.device}")
    print(f"  Resolution  : {cfg.img_height}x{cfg.img_width}")
    print(f"  Mask cible  : {cfg.mask_type}")
    print(f"  Encoder     : {cfg.encoder_name}")
    print(f"  Loss        : {cfg.loss_type}")
    print(f"  Batch size  : {cfg.batch_size}")
    print(f"  Epochs      : {cfg.num_epochs}")
    print(f"  Output      : {run_dir}\n{'='*60}\n")

    torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)
    device = torch.device(cfg.device)

    print("[*] Chargement des donnees...")
    print(f"  Images: {cfg.image_dir}")
    print(f"  Masks : {cfg.mask_dir}")
    print(f"\n  Train (sources: {cfg.train_sources}):")
    train_ds = ECGGridDataset(cfg.image_dir, cfg.mask_dir, cfg.mask_type,
                              cfg.train_sources, cfg.img_height, cfg.img_width, augment=True)
    print(f"  Val   (sources: {cfg.val_sources}):")
    val_ds = ECGGridDataset(cfg.image_dir, cfg.mask_dir, cfg.mask_type,
                            cfg.val_sources, cfg.img_height, cfg.img_width, augment=False)

    if len(train_ds) == 0:
        print("\n[ERREUR] Aucune donnee d'entrainement trouvee."); return

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                              num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)
    val_loader   = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                              num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)

    print("\n[*] Construction du modele...")
    model = smp.Unet(encoder_name=cfg.encoder_name, encoder_weights=cfg.encoder_weights,
                     in_channels=cfg.in_channels, classes=cfg.num_classes, activation=None).to(device)
    print(f"  Parametres: {sum(p.numel() for p in model.parameters()):,}")

    criterion = get_loss(cfg.loss_type, cfg.bce_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=cfg.scheduler_patience, factor=cfg.scheduler_factor)

    history = {k: [] for k in ["train_loss", "val_loss", "train_dice", "val_dice", "train_iou", "val_iou", "lr"]}
    best_val_dice, no_improve = 0, 0

    print(f"\n>>> Debut de l'entrainement ({cfg.num_epochs} epochs)...\n")
    t_start = time.time()
    for epoch in range(1, cfg.num_epochs + 1):
        t0 = time.time()
        lr = optimizer.param_groups[0]["lr"]

        tl, tm = train_one_epoch(model, train_loader, criterion, optimizer, device)
        vl, vm = validate(model, val_loader, criterion, device)
        scheduler.step(vl)

        history["train_loss"].append(tl); history["val_loss"].append(vl)
        history["train_dice"].append(tm["dice"]); history["val_dice"].append(vm["dice"])
        history["train_iou"].append(tm["iou"]);   history["val_iou"].append(vm["iou"])
        history["lr"].append(lr)

        print(f"Epoch {epoch:3d}/{cfg.num_epochs} | "
              f"Train Loss: {tl:.4f}  Dice: {tm['dice']:.3f} | "
              f"Val Loss: {vl:.4f}  Dice: {vm['dice']:.3f}  IoU: {vm['iou']:.3f} | "
              f"LR: {lr:.1e} | {time.time()-t0:.1f}s")

        if vm["dice"] > best_val_dice:
            best_val_dice = vm["dice"]; no_improve = 0
            torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "val_dice": best_val_dice, "config": cfg_dict},
                       os.path.join(run_dir, "checkpoints", "best_model.pth"))
            print(f"  [BEST] Nouveau meilleur modele (Dice: {best_val_dice:.4f})")
        else:
            no_improve += 1

        if epoch % cfg.save_every_n_epochs == 0:
            torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "val_dice": vm["dice"]},
                       os.path.join(run_dir, "checkpoints", f"checkpoint_epoch{epoch:03d}.pth"))

        if epoch % cfg.log_images_every_n_epochs == 0 or epoch == 1:
            model.eval()
            with torch.no_grad():
                si, sm = next(iter(val_loader))
                sp = model(si.to(device)).cpu()
                save_prediction_grid(si, sm, sp,
                    os.path.join(run_dir, "visualizations", f"epoch_{epoch:03d}.png"))

        if no_improve >= cfg.early_stop_patience:
            print(f"\n[STOP] Early stopping apres {cfg.early_stop_patience} epochs sans amelioration"); break

    total_time = time.time() - t_start
    print(f"\n{'='*60}\n  Entrainement termine en {total_time/60:.1f} minutes")
    print(f"  Meilleur Val Dice: {best_val_dice:.4f}")
    print(f"  Resultats dans: {run_dir}\n{'='*60}")
    with open(os.path.join(run_dir, "history.json"), "w") as f:
        json.dump(history, f, indent=2)
    _plot_training_curves(history, os.path.join(run_dir, "training_curves.png"))
    return run_dir


In [ ]:
train(cfg)


In [ ]:
import glob

run_dirs = sorted(glob.glob(os.path.join(cfg.output_dir, "run_*")))
if not run_dirs:
    raise FileNotFoundError(f"Aucun run trouve dans {cfg.output_dir}")
run_dir = run_dirs[-1]
print(f"Run selectionne : {run_dir}")

best_path = os.path.join(run_dir, "checkpoints", "best_model.pth")
ckpts = sorted(glob.glob(os.path.join(run_dir, "checkpoints", "*.pth")))
checkpoint_path = best_path if os.path.exists(best_path) else (ckpts[-1] if ckpts else None)
if checkpoint_path is None:
    raise FileNotFoundError("Aucun checkpoint trouve")
print(f"Checkpoint : {checkpoint_path}")

DEVICE = cfg.device
model = smp.Unet(encoder_name=cfg.encoder_name, encoder_weights=None,
                 in_channels=cfg.in_channels, classes=cfg.num_classes)
checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(DEVICE).eval()
print(f"Modele charge - epoch {checkpoint['epoch']} | Val Dice : {checkpoint.get('val_dice', 'N/A')}")


In [ ]:
image_dir = cfg.image_dir
val_images   = sorted([f for f in os.listdir(image_dir)
                       if f.startswith("ECG_033") and f.endswith(".webp")])
train_images = sorted([f for f in os.listdir(image_dir)
                       if (f.startswith("ECG_031") or f.startswith("ECG_032"))
                       and f.endswith(".webp")])
print(f"Train: {len(train_images)} | Val: {len(val_images)}")


In [ ]:
%matplotlib inline
SET   = "val"
INDEX = 7

image_list = val_images if SET == "val" else train_images
if INDEX >= len(image_list):
    raise IndexError(f"INDEX={INDEX} hors limites - {len(image_list)} images dans '{SET}'")

sample_file = image_list[INDEX]
sample_path = os.path.join(image_dir, sample_file)
mask_path   = os.path.join(cfg.mask_dir, sample_file.replace(".webp", ""), cfg.mask_type)

print(f"Image : {sample_file}")
print(f"Mask  : {mask_path}  (existe={os.path.exists(mask_path)})")
if not os.path.exists(mask_path):
    raise FileNotFoundError(mask_path)

img_pil = Image.open(sample_path).convert("RGB")
img_pil = img_pil.resize((cfg.img_width, cfg.img_height), Image.BILINEAR)
img_np  = np.array(img_pil, dtype=np.float32) / 255.0

mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
img_norm = (img_np - mean) / std

mask_pil = Image.open(mask_path).convert("L")
mask_pil = mask_pil.resize((cfg.img_width, cfg.img_height), Image.NEAREST)
mask_np  = np.array(mask_pil, dtype=np.float32) / 255.0

img_tensor = torch.from_numpy(img_norm).permute(2, 0, 1).unsqueeze(0).float().to(DEVICE)
with torch.no_grad():
    output = model(img_tensor)
    prediction = torch.sigmoid(output)
pred_np   = prediction.squeeze().cpu().numpy()
pred_bin  = (pred_np > 0.5).astype(np.float32)

inter = (pred_bin * mask_np).sum()
dice  = (2 * inter) / (pred_bin.sum() + mask_np.sum() + 1e-8)
print(f"\nPrediction continue - max={pred_np.max():.3f}, mean={pred_np.mean():.4f}")
print(f"Dice (seuil 0.5) sur cette image : {dice:.4f}")

overlay = img_np.copy()
tp = (pred_bin > 0.5) & (mask_np > 0.5)
fp = (pred_bin > 0.5) & (mask_np < 0.5)
fn = (pred_bin < 0.5) & (mask_np > 0.5)
overlay[tp] = [0, 1, 0]; overlay[fp] = [1, 0, 0]; overlay[fn] = [0, 0, 1]

fig, axes = plt.subplots(1, 4, figsize=(22, 6))
fig.suptitle(f"[{SET.upper()}] {sample_file} - Dice: {dice:.4f}", fontsize=12)
axes[0].imshow(img_np);                                                     axes[0].set_title("Image augmentee");                       axes[0].axis("off")
axes[1].imshow(mask_np,  cmap="gray", vmin=0, vmax=1);                      axes[1].set_title("Intersections GT");                       axes[1].axis("off")
axes[2].imshow(pred_np,  cmap="hot",  vmin=0, vmax=max(pred_np.max(),1e-6)); axes[2].set_title(f"Pred sigmoid (max={pred_np.max():.2f})"); axes[2].axis("off")
axes[3].imshow(overlay);                                                    axes[3].set_title("Overlay (V=TP, R=FP, B=FN)");             axes[3].axis("off")
plt.tight_layout(); plt.show(); plt.close()
